# Phase-4→Phase-5 bridge — freq-weighted Benna-Fusi α sweep (Colab)

**Purpose.** Test brainstorm idea 5 (retrieval-frequency-weighted α coupling in the Benna-Fusi cascade) at the canonical Phase 4 graduation regime — n=10 seeds, n_cues=3000, online Hebbian, A_k off, m=6.

**Hypothesis / what we measure.** The headline drill-down `corr_u_m_retrieval_count` in `phase34_results.json[results].phase3_phase4[*].death_diag[*]` measures the Pearson correlation between a pattern's retrieval count and its slow-variable u_m at the final checkpoint. Under λ=0 this is the natural correlation that comes from u_1 input scaling with retrieval count. Under λ>0 the per-pattern α_eff also scales with retrieval count, so the correlation either rises (transient-dominated) or falls (steady-state-dominated where the high-α boundary leak overtakes input). See [experiment plan](../notes/notes/2026-05-16-freq-weighted-alpha-experiment-plan.md) and the steady-state asymmetry test in `tests/test_phase4_consolidation.py::TestFreqWeightedAlpha`.

**4 conditions × 10 seeds = 40 runs:**
- `lam0p0` (λ=0.0)  — sanity replicate of [report 038](../reports/038_phase4_d1_graduation.md). Should reproduce its numbers seed-by-seed.
- `lam0p5` (λ=0.5)  — modest freq-weighting; max-count atom gets α_eff = 0.375.
- `lam1p0` (λ=1.0)  — strong freq-weighting; max-count atom gets α_eff = 0.5 (at the m=6 CFL stability bound).
- `lam2p0` (λ=2.0)  — supercritical stress; max-count atom gets α_eff = 0.75. Tests where stability breaks.

**Other params identical to report 038:** online Hebbian, st=0.3, n_cues=3000, A_k off, death threshold=0.05, death window=10, no-reencode-discovered.

**Required local patch:** `experiments/19_phase34_integrated.py` must emit per-scale `corr_u_m_retrieval_count` / `gini_u_m` in death_diag, and accept `--alpha-freq-lambda`. Committed in commit 7822301; cell 1 verifies.

In [ ]:
# 1. Clone the repo and verify the freq-weighted-α patch is present.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git log --oneline -5

# Patch sanity check
import subprocess
for marker, fpath, label in [
    ('alpha_freq_lambda',         'src/energy_memory/phase4/consolidation.py', 'ConsolidationConfig field'),
    ('corr_u_m_retrieval_count',  'experiments/19_phase34_integrated.py',       'death_diag drill-down'),
    ('alpha-freq-lambda',         'experiments/19_phase34_integrated.py',       'CLI flag'),
]:
    n = int(subprocess.check_output(['grep', '-c', marker, fpath]).decode().strip())
    assert n >= 1, f'{label} ({marker} in {fpath}) missing — abort!'
    print(f'  ok: {label} ({marker}) x{n} in {fpath}')

In [ ]:
# 2. Mount Drive and copy the Phase 3c codebook into place.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps.
!pip install -q datasets py-spy

In [ ]:
# 4. CPU-ONLY sanity check. Parent NEVER touches CUDA so workers can grab the GPU.
import os
print('cpu count:', os.cpu_count())

import sys; sys.path.insert(0, 'src')
from energy_memory.phase2.persistence import load_codebook
cb = load_codebook('reports/phase3c_reconstruction/phase3c_codebook_reconstruction.pt', device='cpu')
print('codebook (parent CPU load):', cb.shape, cb.dtype, cb.device)

# Verify the new freq-weighted-α mechanism behaves correctly without touching CUDA.
from energy_memory.phase4.consolidation import ConsolidationConfig, ConsolidationState
s = ConsolidationState(ConsolidationConfig(m=4, alpha=0.25, alpha_freq_lambda=1.0), device='cpu')
s.add_pattern(novelty_strength=1.0); s.add_pattern(novelty_strength=1.0)
s.reinforce(0); s.reinforce(0); s.reinforce(0); s.reinforce(0); s.reinforce(0)
s.reinforce(1)
s.step_dynamics()
# pattern 0 (5 retrievals) should have larger u_2 than pattern 1 (1 retrieval) under λ>0
assert s.u[0, 1].item() > s.u[1, 1].item(), 'α_eff scaling broken!'
print(f'  freq-weighted α one-step check: u_2[0]={s.u[0, 1]:.4f} > u_2[1]={s.u[1, 1]:.4f}  OK')

del cb, s
import gc; gc.collect()

In [ ]:
# 4b. Pre-warm the wikitext cache so all subprocesses don't race on download.
print('warming wikitext cache...')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
splits = load_corpus_splits('wikitext', Path('.'), wikitext_name='wikitext-2-raw-v1')
print('  train:', len(splits['train']), 'rows')
print('  validation:', len(splits['validation']), 'rows')
print('cache warmed.')
del splits; gc.collect()

In [ ]:
# 4c. GPU info (no CUDA init).
print('=== GPU info ===')
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
print()
print('=== Current GPU processes (should be empty before launching workers) ===')
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

In [ ]:
# 5. Sweep launcher. Runs 4 λ values × 10 seeds = 40 jobs.
# Strategy: one batch per λ (10 seeds in parallel each), so a slow batch can
# be interrupted without losing the other λ values. Each batch ~14 min wall.

PARALLEL_BATCH_SIZE = 10
LAMBDAS = [
    ('lam0p0', 0.0),
    ('lam0p5', 0.5),
    ('lam1p0', 1.0),
    ('lam2p0', 2.0),
]
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]

import subprocess, os, time, signal
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
RUN_TAG_BASE = 'freqalpha_n3k'
log_root = Path(f'reports/phase34_{RUN_TAG_BASE}_colab')
log_root.mkdir(parents=True, exist_ok=True)

def launch(seed, lam_tag, lam_val):
    out_dir = f'reports/phase34_{RUN_TAG_BASE}_{lam_tag}_seed{seed}'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'{lam_tag}_seed{seed}.log'
    logf = open(log_path, 'w')
    proc = subprocess.Popen(
        ['python', 'experiments/19_phase34_integrated.py',
         '--updater-kind', 'hebbian',
         '--device', 'cuda',
         '--seed', str(seed),
         '--n-cues', '3000',
         '--checkpoint-every', '500',
         '--success-threshold', '0.3',
         '--death-threshold', '0.05',
         '--death-window', '10',
         '--inhibition-gain', '0.0',
         '--inhibition-decay', '0.0',
         '--alpha-freq-lambda', str(lam_val),
         '--no-reencode-discovered',
         '--output-dir', out_dir],
        stdout=logf, stderr=subprocess.STDOUT,
    )
    return proc, logf, out_dir

def kill_all(procs_dict):
    for k, (proc, logf, out_dir) in procs_dict.items():
        try:
            proc.send_signal(signal.SIGKILL)
            logf.close()
        except Exception:
            pass

def snapshot(procs_dict, t0, lam_tag):
    elapsed = (time.time() - t0) / 60
    print(f'  --- snapshot at {elapsed:.1f} min ({lam_tag}) ---')
    try:
        gpu = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,utilization.gpu', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip()
        print(f'  GPU: {gpu}')
        gpu_p = subprocess.check_output(
            ['nvidia-smi', '--query-compute-apps=pid,used_memory', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip()
        n_gpu_procs = len([l for l in gpu_p.splitlines() if l.strip()])
        print(f'  GPU processes attached: {n_gpu_procs}')
    except Exception as e:
        print(f'  GPU snapshot failed: {e}')
    for k, (proc, logf, out_dir) in procs_dict.items():
        log = log_root / f'{k}.log'
        size = log.stat().st_size if log.exists() else 0
        try:
            line = subprocess.check_output(['tail', '-1', str(log)],
                stderr=subprocess.DEVNULL).decode().strip()[:90]
        except Exception:
            line = ''
        print(f'  {k:>22}: {size:>7}b  {line}')

def run_batch_for_lambda(lam_tag, lam_val, t0):
    print(f'\n=== λ={lam_val} ({lam_tag}) — launching {len(SEEDS)} seeds ({time.strftime("%H:%M:%S")}) ===')
    procs = {f'{lam_tag}_seed{seed}': launch(seed, lam_tag, lam_val) for seed in SEEDS}
    print(f'  pids={[p[0].pid for p in procs.values()]}')
    remaining = dict(procs)
    poll_count = 0
    try:
        while remaining:
            done_this_round = []
            for k, (proc, logf, out_dir) in remaining.items():
                rc = proc.poll()
                if rc is not None:
                    logf.close()
                    elapsed = (time.time() - t0) / 60
                    ok = 'OK' if rc == 0 else f'FAILED (exit={rc})'
                    json_exists = Path(out_dir, 'phase34_results.json').exists()
                    print(f'  [{elapsed:5.1f} min] {k}: {ok}  json={json_exists}')
                    done_this_round.append(k)
            for k in done_this_round:
                del remaining[k]
            if remaining:
                poll_count += 1
                if poll_count % 3 == 0:
                    snapshot(remaining, t0, lam_tag)
                time.sleep(30)
    except KeyboardInterrupt:
        print('\n!!! Interrupted — SIGKILLing all remaining workers !!!')
        kill_all(remaining)
        raise

t0 = time.time()
for lam_tag, lam_val in LAMBDAS:
    run_batch_for_lambda(lam_tag, lam_val, t0)

print(f'\nALL {len(LAMBDAS)*len(SEEDS)} RUNS DONE in {(time.time()-t0)/60:.1f} min')
print()
print('=== Final GPU state ===')
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
# 5c. EMERGENCY: kill all workers if cell 5 misbehaves. Runtime → Interrupt cell 5 first, THEN run this.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'experiments/19' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL)
            print(f'  killed {pid}')
            killed += 1
        except Exception as e:
            print(f'  {pid}: {e}')
print(f'killed {killed} workers')

In [ ]:
# 6. Copy results back to Drive.
import shutil, os
RUN_TAG_BASE = 'freqalpha_n3k'
LAMBDAS = [('lam0p0', 0.0), ('lam0p5', 0.5), ('lam1p0', 1.0), ('lam2p0', 2.0)]
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)
for lam_tag, _ in LAMBDAS:
    for seed in SEEDS:
        src = f'reports/phase34_{RUN_TAG_BASE}_{lam_tag}_seed{seed}'
        if os.path.isdir(src):
            shutil.copytree(src, f'{dst}/phase34_{RUN_TAG_BASE}_{lam_tag}_seed{seed}', dirs_exist_ok=True)
shutil.copytree(f'reports/phase34_{RUN_TAG_BASE}_colab',
                f'{dst}/phase34_{RUN_TAG_BASE}_colab', dirs_exist_ok=True)
print('results in', dst)
!ls {dst} | grep -E 'freqalpha_n3k' | head -20

In [ ]:
# 7. Per-λ summary across seeds. Headline: corr_u_m_retrieval_count and gini_u_m.
import json, math
from pathlib import Path
import statistics as st

RUN_TAG_BASE = 'freqalpha_n3k'
LAMBDAS = [('lam0p0', 0.0), ('lam0p5', 0.5), ('lam1p0', 1.0), ('lam2p0', 2.0)]
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
T95 = {9: 2.262, 8: 2.306, 7: 2.365, 6: 2.447, 5: 2.571, 4: 2.776}

def load(lam_tag, seed):
    p = Path(f'reports/phase34_{RUN_TAG_BASE}_{lam_tag}_seed{seed}/phase34_results.json')
    if not p.exists():
        return None
    return json.loads(p.read_text())

def t_ci(values):
    n = len(values)
    if n < 2: return st.fmean(values) if values else 0.0, 0.0, 0.0
    m_ = st.fmean(values)
    v = sum((x - m_) ** 2 for x in values) / (n - 1)
    se = math.sqrt(v / n)
    t = T95.get(n - 1, 1.96)
    return m_, m_ - t*se, m_ + t*se

agg = {}
for lam_tag, lam_val in LAMBDAS:
    rows = []
    for seed in SEEDS:
        d = load(lam_tag, seed)
        if d is None:
            continue
        c = d['results']['phase3_phase4'][-1]
        a = d['results']['baseline_static'][-1]
        dd = c.get('death_diag', {}).get('2', {})
        rows.append({
            'seed': seed,
            'R10_A': a['topk'], 'R10_C': c['topk'],
            'top1_A': a['top1'], 'top1_C': c['top1'],
            'capt5_A': a['cap_t_05'], 'capt5_C': c['cap_t_05'],
            'ms_w3_A': a.get('meta_stable_w3', float('nan')),
            'ms_w3_C': c.get('meta_stable_w3', float('nan')),
            'corr_u_m_rc': dd.get('corr_u_m_retrieval_count', float('nan')),
            'gini_u_m':    dd.get('gini_u_m', float('nan')),
            'rc_max':      dd.get('retrieval_count_max', 0),
            'rc_nonzero':  dd.get('retrieval_count_nonzero', 0),
            'deaths':      c.get('deaths_total', 0),
            'n_pat_w2':    c.get('n_patterns_alive_w2', 0),
        })
    agg[lam_tag] = rows

print(f"{'λ':>6}  {'n':>3}  {'corr_u_m_rc':>18}  {'gini_u_m':>16}  {'Δms_w3 (CA)':>15}  {'ΔR@10':>15}")
for lam_tag, lam_val in LAMBDAS:
    rows = agg.get(lam_tag, [])
    if not rows:
        print(f"  λ={lam_val:.2f}: no data")
        continue
    corrs = [r['corr_u_m_rc'] for r in rows if not math.isnan(r['corr_u_m_rc'])]
    ginis = [r['gini_u_m']    for r in rows if not math.isnan(r['gini_u_m'])]
    dms3  = [r['ms_w3_C'] - r['ms_w3_A'] for r in rows if not (math.isnan(r['ms_w3_A']) or math.isnan(r['ms_w3_C']))]
    dR10  = [r['R10_C']  - r['R10_A']  for r in rows]
    c_m, c_l, c_h = t_ci(corrs)
    g_m, g_l, g_h = t_ci(ginis)
    m_m, m_l, m_h = t_ci(dms3)
    r_m, r_l, r_h = t_ci(dR10)
    print(f"  {lam_val:>5.2f}  {len(rows):>3}  "
          f"{c_m:>+6.3f} [{c_l:>+5.2f},{c_h:>+5.2f}]  "
          f"{g_m:>+5.3f} [{g_l:>+4.2f},{g_h:>+4.2f}]  "
          f"{m_m:>+5.3f} [{m_l:>+5.2f},{m_h:>+5.2f}]  "
          f"{r_m:>+5.3f} [{r_l:>+5.2f},{r_h:>+5.2f}]")

# Persist aggregate for downstream report.
outp = Path(f'reports/phase34_{RUN_TAG_BASE}_aggregate.json')
outp.write_text(json.dumps({'lambdas': LAMBDAS, 'seeds': SEEDS, 'per_lambda': agg}, indent=2))
print(f'\naggregate saved -> {outp}')

In [ ]:
# 7b. Sanity check: λ=0.0 should reproduce report 038's seed-by-seed numbers.
# If lam0p0's R@10 and ms_w3 don't match report 038 exactly, the bridge
# experiment's λ=0 is not a true control and the comparisons are invalid.
REPORT_038_LAM0 = {
    # seed: (R@10_C, ms_w3_C)
    17: (0.3422, 0.000), 11: (0.3714, 0.000), 23: (0.3493, 0.000), 1: (0.3656, 0.000),
    2: (0.2955, 0.000), 3: (0.3624, 0.000), 5: (0.2757, 0.000),  7: (0.3707, 0.000),
    13: (0.3302, 0.000), 19: (0.3184, 0.000),
}
print('Sanity: λ=0.0 vs report 038 final-checkpoint values')
print(f"  {'seed':>4}  {'this R@10':>10}  {'038 R@10':>9}  {'Δ':>7}  match?")
fails = 0
for r in agg.get('lam0p0', []):
    seed = r['seed']
    if seed not in REPORT_038_LAM0:
        continue
    exp_R, exp_ms = REPORT_038_LAM0[seed]
    delta = r['R10_C'] - exp_R
    ok = abs(delta) < 0.005  # tolerance for any harmless determinism wiggle
    if not ok:
        fails += 1
    print(f"  {seed:>4}  {r['R10_C']:>10.4f}  {exp_R:>9.4f}  {delta:>+7.4f}  {'ok' if ok else 'FAIL'}")
if fails == 0:
    print('λ=0.0 replicates report 038 within tolerance.')
else:
    print(f'{fails} seeds disagree with report 038. Investigate before trusting other-λ results.')